In [7]:
import os, sys
from pathlib import Path
_p = Path.cwd().resolve()
while not (_p / "src" / "pipeline" / "config.py").is_file():
    if _p.parent == _p:
        raise SystemExit("please run this notebook from inside the repository")
    _p = _p.parent
sys.path.insert(0, str(_p / "src"))

FOLD = 0  # 0-4, or None = all five folds

import numpy as np
import torch
from pipeline import config
from pipeline.test import casee_eval as ce
DEV = ce.device()
FOLDS = range(5) if FOLD is None else [FOLD]


In [8]:
from torch.utils.data import DataLoader


LIBS = ["vakulskas_hek", "vakulskas_emx", "choi_sg8"]

scores = {}
for k in FOLDS:
    net = ce.load_sim(k, DEV)
    print("--- fold %d" % k)
    for lib in LIBS:
        ds = ce.mut_dataset(lib)
        y, z = [], []
        for seq, mut, eff, c, p in DataLoader(ds, batch_size=1):
            if "C" in mut[0] or "WT" in mut[0]:
                continue
            with torch.no_grad():
                e = net(c.to(DEV), p.to(DEV))
            y.append(eff.item())
            z.append(e.item())
        key = "%s_fold%d" % (lib, k)
        scores[key] = ce.rho(y, z)
        print("%-16s n=%d  Spearman = %.4f" % (lib, len(y), scores[key]))


--- fold 0
vakulskas_hek    n=13  Spearman = 0.6025
vakulskas_emx    n=13  Spearman = 0.8619
choi_sg8         n=7  Spearman = 0.8214


In [9]:
print()
for lib in LIBS:
    v = [r for n, r in scores.items() if n.startswith(lib + "_fold")]
    if len(v) > 1:
        print("%-16s mean = %.4f  std = %.4f  (folds=%d)" % (lib, np.mean(v), np.std(v), len(v)))
